# Session 47: Saving and Loading Models

Data Science with Python, Level 3, Stage 7: Deployment

## Learning Objectives

By the end of this session you will be able to:

1. Explain why fitted models are saved.
2. Save and load a pipeline with joblib.
3. Record versions and metrics in a metadata file.
4. Write a prediction function for one input.
5. Validate inputs before predicting.
6. Know the risks of loading model files.

## Where We Are

Until now every notebook trained its model from scratch, every time. A web app cannot do that: it must answer in a fraction of a second, and it should use exactly the model you tested. So you train once, **save** the fitted pipeline to a file, and **load** it wherever predictions are needed.

Today you build the engine of the Stage 7 app: the final apartment price pipeline, saved with `joblib`, a metadata file that records how it was made, and a `predict_price` function that checks its input before predicting. In Session 48 a Streamlit app will call that function.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import json
from datetime import date
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
flats = pd.read_csv(DATA_URL + "apartments.csv")
X = flats.drop(columns=["apartment_id", "price_k"])
y = flats["price_k"]

## 1. The Final Model

In [ ]:
numeric = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
preprocess = ColumnTransformer([
    ("district", OneHotEncoder(handle_unknown="ignore"), ["district"]),
    ("numbers", StandardScaler(), numeric),
    ("flags", "passthrough", ["elevator", "parking"]),
])
model = Pipeline([("prep", preprocess), ("model", LinearRegression())])

In [ ]:
folds = KFold(n_splits=5, shuffle=True, random_state=0)
cv = cross_validate(model, X, y, cv=folds,
                    scoring=["neg_mean_absolute_error", "r2"])
mae = -cv["test_neg_mean_absolute_error"].mean()
r2 = cv["test_r2"].mean()
print(f"CV MAE {mae:.1f} thousand, CV R2 {r2:.2f}")

In [ ]:
model.fit(X, y)
print(model.predict(X.head(3)).round(1))
print(y.head(3).values)

## 2. Saving and Loading

In [ ]:
Path("models").mkdir(exist_ok=True)
joblib.dump(model, "models/price_model.joblib")

size_kb = Path("models/price_model.joblib").stat().st_size / 1024
print(f"Saved, {size_kb:.1f} KB")

In [ ]:
loaded = joblib.load("models/price_model.joblib")

same = np.allclose(loaded.predict(X), model.predict(X))
print("Same predictions:", same)
print(loaded.named_steps["prep"].named_transformers_["district"].categories_)

## 3. Metadata

In [ ]:
metadata = {
    "model_file": "price_model.joblib",
    "trained_on": str(date.today()),
    "rows": len(X),
    "features": list(X.columns),
    "districts": sorted(X["district"].unique()),
    "cv_mae_thousands": round(mae, 1),
    "cv_r2": round(r2, 3),
    "sklearn_version": sklearn.__version__,
}
Path("models/metadata.json").write_text(json.dumps(metadata, indent=2))
print(list(metadata))

## 4. A Prediction Function

In [ ]:
def predict_price(model, area_m2, rooms, floor, building_age,
                  distance_center_km, district, elevator, parking):
    row = pd.DataFrame([{
        "district": district, "area_m2": area_m2, "rooms": rooms,
        "floor": floor, "building_age": building_age,
        "distance_center_km": distance_center_km,
        "elevator": elevator, "parking": parking}])
    return float(model.predict(row)[0])

In [ ]:
price = predict_price(loaded, area_m2=85, rooms=3, floor=4, building_age=10,
                      distance_center_km=3.5, district="Riverside",
                      elevator=True, parking=True)
print(f"Estimated price: {price:.0f} thousand")

In [ ]:
def check_input(area_m2, rooms, district, districts):
    problems = []
    if not 20 <= area_m2 <= 250:
        problems.append("Area must be between 20 and 250 square metres.")
    if not 1 <= rooms <= 8:
        problems.append("Rooms must be between 1 and 8.")
    if district not in districts:
        problems.append(f"Unknown district: {district}.")
    return problems

print(check_input(85, 3, "Riverside", metadata["districts"]))
print(check_input(900, 0, "Airport", metadata["districts"]))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Saving only the model | `joblib.dump(model["model"], ...)` without preprocessing | Save the whole pipeline. |
| Version mismatch | Training in one scikit-learn version, loading in another | Pin the version in requirements.txt. |
| Unknown files | Loading a .pkl file found online | Only load model files you created or trust. |
| No checks | Predicting for impossible inputs | Validate inputs and show clear messages. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Save the Churn Model

Fit the churn logistic regression pipeline from Session 44 on all customers, save it as `models/churn_model.joblib`, load it back and check that the predicted probabilities match.

In [ ]:
# Write your code below

### In-Class Exercise 2: Read the Metadata

Load `models/metadata.json` with `json.loads(Path(...).read_text())` and print a one-line summary: model file, number of rows, CV MAE and scikit-learn version.

In [ ]:
# Write your code below

### In-Class Exercise 3: Price Range

Write `price_range(price, mae)` that returns a text like `between 120 and 140 thousand`, rounding to the nearest 5. Use it with the Riverside example.

In [ ]:
# Write your code below

### In-Class Exercise 4: Many Apartments at Once

Create a DataFrame of five new apartments (your own values, all eight input columns), predict their prices with the loaded model in one call, and add the predictions as a column.

In [ ]:
# Write your code below

## Summary

- Train once, save the fitted pipeline, load it wherever needed.
- `joblib.dump` and `joblib.load` save and restore the whole pipeline.
- A metadata JSON records features, metrics and versions.
- A prediction function gives the app one clean entry point.
- Validate inputs and give clear messages.
- Only load model files you trust, with matching versions.

## What's Next

Next session you will build a web page around `predict_price`: sliders and drop-downs for the inputs, a button, and the estimated price, all in about 40 lines of Python with Streamlit.